# Алмазов МРТ — финальная оценка (версия БЕЗ видеокарты)
Работает на обычном CPU: модель уже обучена, её ответы на тест сохранены в Drive (`almazov/exp_probs`). **Среда выполнения → Выполнить все** (~3–5 минут).
Считает для модели, выбранной по валидации, все метрики основной работы и готовит её веса для демо (`almazov/final/`).
Разбиение — то же (`split_v3.csv`); разрешение снимков читается из заголовков файлов в архиве, без распаковки.

In [ ]:
# === 0. НАСТРОЙКА (без видеокарты) ===
from google.colab import drive
drive.mount('/content/drive')
import os, json, glob, zipfile, shutil
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
W = Path('/content/drive/MyDrive/almazov'); (W/'figs').mkdir(exist_ok=True)
CLASSES = ['Глиома', 'Менингиома', 'Гипофиз', 'Опухоли нет']
ORG = Path('/content/org')   # пути в split_v3.csv записаны относительно этой папки (как в основном блокноте)

# --- разбиение: тот же файл, что в основной работе ---
df = pd.read_csv(W/'split_v3.csv', keep_default_na=False)
df['pid'] = df.pid.astype(str); df['test_unique'] = df.test_unique.astype(str).eq('True')
LAB = df.label.values.astype(int)
df['label_conflict'] = df.group.map(df.groupby('group').label.nunique() > 1).values   # то же определение, что в основном блокноте
use = ~df.label_conflict.values
TE = np.where((df.part == 'test').values & use)[0]; VA = np.where((df.part == 'val').values & use)[0]
TEU = np.where(df.test_unique.values & use)[0]
print('Снимков:', len(df), '| тест:', len(TE), '| валидация:', len(VA), '| уникальных в тесте:', len(TEU))

# --- разрешение снимков из архива (только заголовки, без распаковки) ---
zs = sorted(glob.glob('/content/drive/MyDrive/**/Almazov_MRI*.zip', recursive=True)); assert zs, 'Не нашёл Almazov_MRI.zip в Google Drive'
size = {}
with zipfile.ZipFile(zs[0]) as z:
    for info in z.infolist():
        name = info.filename
        if not (info.flag_bits & 0x800):
            try: name = name.encode('cp437').decode('cp866')
            except Exception: pass
        parts = name.replace('\\', '/').split('/')
        if parts and parts[0].startswith('Almazov_MRI'): parts = parts[1:]
        if info.is_dir() or not parts or not parts[-1].lower().endswith(('.jpg', '.jpeg', '.png')): continue
        with z.open(info) as f: w_, h_ = Image.open(f).size
        size[str(ORG.joinpath(*parts))] = (h_, w_)
miss = int((~df.path.isin(size)).sum()); print('Размеры прочитаны:', len(size), '| не найдено в архиве:', miss)
df['h'] = df.path.map(lambda p: size.get(p, (0, 0))[0]); df['w'] = df.path.map(lambda p: size.get(p, (0, 0))[1])

In [ ]:
# === F1. ФИНАЛЬНАЯ ОЦЕНКА ВЫБРАННОЙ МОДЕЛИ ===
from sklearn.metrics import f1_score, balanced_accuracy_score, accuracy_score, roc_auc_score, confusion_matrix
R = json.load(open(W/'experiments_v1.json')); PROB = W/'exp_probs'
sel = R['selected']['tag']; members = R[sel].get('members', [sel])
print('Выбрана по валидации:', sel, '| состав:', members, '| val macro-F1', round(R['selected']['val_macro_f1'], 4))
Z = [np.load(PROB/f'{k}.npz') for k in members]
Pva, Pte = np.mean([z['va'] for z in Z], 0), np.mean([z['te'] for z in Z], 0)
assert len(Pte) == len(TE) and len(Pva) == len(VA), f'Размеры не совпадают с разбиением: {len(Pte)} vs {len(TE)}'
assert abs(float((Pte.argmax(1) == LAB[TE]).mean()) - R[sel]['test']['accuracy']) < 1e-6, 'Порядок снимков не совпал с экспериментами'

def evaluate(P, Y):
    pr = P.argmax(1); tt = Y != 3; pt = 1 - P[:, 3]
    return dict(n=int(len(Y)), accuracy=float(accuracy_score(Y, pr)), macro_f1=float(f1_score(Y, pr, average='macro')),
                balanced_acc=float(balanced_accuracy_score(Y, pr)),
                recall={CLASSES[k]: (float((pr[Y == k] == k).mean()) if (Y == k).any() else None) for k in range(4)},
                tumor_sens=float(((pr != 3) & tt).sum() / max(tt.sum(), 1)), tumor_spec=float(((pr == 3) & ~tt).sum() / max((~tt).sum(), 1)),
                tumor_auc=(float(roc_auc_score(tt, pt)) if 0 < tt.sum() < len(Y) else None))
def boot(fn, groups, n=1000, seed=0):   # тот же бутстрэп по группам, что в основной работе
    rs2 = np.random.RandomState(seed); ug = np.unique(groups); by = {g: np.where(groups == g)[0] for g in ug}; vals = []
    for _ in range(n):
        ii = np.concatenate([by[g] for g in rs2.choice(ug, len(ug), replace=True)]); vals.append(fn(ii))
    return [round(float(x), 4) for x in np.percentile(vals, [2.5, 97.5])]

Y = LAB[TE]; pr = Pte.argmax(1); gT = df.group.values[TE]
F = {'selected': sel, 'members': members, 'val_macro_f1': R['selected']['val_macro_f1'], 'test_group': evaluate(Pte, Y)}
F['ci95'] = dict(accuracy=boot(lambda ii: (pr[ii] == Y[ii]).mean(), gT),
                 macro_f1=boot(lambda ii: f1_score(Y[ii], pr[ii], average='macro'), gT),
                 tumor_sens=boot(lambda ii: ((pr[ii] != 3) & (Y[ii] != 3)).sum() / max((Y[ii] != 3).sum(), 1), gT),
                 tumor_spec=boot(lambda ii: ((pr[ii] == 3) & (Y[ii] == 3)).sum() / max((Y[ii] == 3).sum(), 1), gT))
pos = {i: k for k, i in enumerate(TE)}
F['test_group_unique'] = evaluate(Pte[[pos[i] for i in TEU]], LAB[TEU])
fp_ = df.pid.values[TE] != ''
F['figshare_new_patients'] = evaluate(Pte[fp_], Y[fp_]); F['other_subset'] = evaluate(Pte[~fp_], Y[~fp_])
# режим скрининга — тот же способ выбора порога, что в основной работе (по валидации, не выше 0.5)
pv = 1 - Pva[:, 3]; thr = float(min(0.5, np.quantile(pv[LAB[VA] != 3], 0.01)))
pt_ = 1 - Pte[:, 3]; tt_ = Y != 3; fl = pt_ >= thr
F['screening'] = dict(thr=thr, sens=float((fl & tt_).sum() / tt_.sum()), spec=float((~fl & ~tt_).sum() / (~tt_).sum()))
t_ = df.iloc[TE].copy(); t_['ok'] = pr == Y
t_['res'] = pd.cut(t_[['h', 'w']].max(1), [0, 256, 400, 100000], labels=['до 256', '257-400', 'больше 400'])
F['by_resolution'] = {str(k): dict(acc=float(v['mean']), n=int(v['size'])) for k, v in t_.groupby('res', observed=True).ok.agg(['mean', 'size']).iterrows()}
F['confusion_matrix'] = confusion_matrix(Y, pr, labels=range(4)).tolist()
F['base_for_comparison'] = R['base']['test']
F['selected_seed_summary'] = R.get('selected_seed_summary')  # повторы — только если считались на видеокарте
json.dump(F, open(W/'results_final.json', 'w'), ensure_ascii=False, indent=1)
print(json.dumps({k: v for k, v in F.items() if k not in ('base_for_comparison',)}, ensure_ascii=False, indent=1))

cm = np.array(F['confusion_matrix'])
fig, ax = plt.subplots(figsize=(5.2, 4.4)); ax.imshow(cm, cmap='Blues')
for i in range(4):
    for j in range(4): ax.text(j, i, cm[i, j], ha='center', va='center', color='white' if cm[i, j] > cm.max() / 2 else 'black')
ax.set_xticks(range(4), CLASSES, rotation=25); ax.set_yticks(range(4), CLASSES)
ax.set_xlabel('Предсказание модели'); ax.set_ylabel('Истина'); plt.tight_layout()
plt.savefig(W/'figs'/'cm_final.png', dpi=150); plt.show()

In [ ]:
# === F2. ВЕСА ВЫБРАННОЙ МОДЕЛИ ДЛЯ ДЕМО (части по 20 МБ — для загрузки на GitHub) ===
import hashlib
FD = W/'final'; shutil.rmtree(FD, ignore_errors=True); FD.mkdir()
manifest = []
for k in members:
    src_ = W/'ckpt'/('cls_v3_best.pt' if k == 'base' else f'exp_{k}_best.pt')
    arch = R[k].get('arch', 'efficientnet_b0'); size = R[k].get('size', 256)
    data = open(src_, 'rb').read(); parts = []
    for i in range(0, len(data), 20 * 1024 * 1024):
        nm = f'cls_{k}.pt.part{i // (20 * 1024 * 1024)}'; open(FD/nm, 'wb').write(data[i:i + 20 * 1024 * 1024]); parts.append(nm)
    manifest.append(dict(tag=k, arch=arch, size=size, parts=parts, sha256=hashlib.sha256(data).hexdigest()))
    print(f'{k}: {arch}, {len(data) / 1e6:.1f} МБ -> {len(parts)} частей')
json.dump(dict(selected=sel, models=manifest, screen_thr=F['screening']['thr']), open(FD/'cls_models.json', 'w'), indent=1)
print('Готово: папка almazov/final в Google Drive. Скачай её целиком (правый клик -> Скачать) в папку MRI_Downloader и напиши мне.')